In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random as rd

In [2]:
drive = pd.read_csv('dataset.csv').sample(400000)
drive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 400000 entries, 321021 to 54834
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   DriverId    400000 non-null  int64  
 1   EventName   400000 non-null  object 
 2   Latitude    400000 non-null  float64
 3   Longitude   400000 non-null  float64
 4   Speed km/h  400000 non-null  float64
 5   ts          400000 non-null  object 
dtypes: float64(3), int64(1), object(2)
memory usage: 21.4+ MB


In [3]:
drive.describe()

,DriverId,Latitude,Longitude,Speed km/h
count,400000.000000,400000.000000,400000.000000,400000.000000
mean,43.603952,34.150445,-118.170184,63.190250
std,26.453084,0.448524,0.164940,39.038804
min,0.000000,31.552999,-118.662188,0.000000
25%,19.000000,33.940629,-118.284688,30.000000
50%,45.000000,34.104752,-118.209777,65.000000
75%,65.000000,34.428584,-118.085809,96.000000
max,88.000000,35.248460,-117.442790,176.000000


In [4]:
drive = drive.drop_duplicates()
drive = drive.drop(drive[drive['Speed km/h'] < 5].index)

In [5]:
drive.sample(10)

,DriverId,EventName,Latitude,Longitude,Speed km/h,ts
449090,45,Distance Event,33.334854,-118.463243,39.0,2017-11-16 19:02:07.920
809632,81,Distance Event,33.838015,-118.393520,113.0,2017-11-10 22:05:01.840
795617,78,Distance Event,34.315353,-118.215938,19.0,2017-11-16 17:15:13.100
644776,62,Timed Event,34.931589,-117.972163,29.0,2017-11-07 07:46:57.380
36409,1,Distance Event,34.138477,-118.197739,71.0,2017-11-17 15:00:12.940
595207,55,Distance Event,34.638077,-117.805285,85.0,2017-11-12 11:50:44.540
76851,5,Timed Event,34.343085,-118.232112,20.0,2017-11-15 15:12:09.400
140094,12,Distance Event,33.361395,-118.446867,107.0,2017-11-07 09:55:25.080
275895,25,Distance Event,34.088496,-118.311896,39.0,2017-11-12 15:57:04.780
844767,85,Distance Event,33.944929,-118.299293,44.0,2017-11-08 17:44:47.380


In [6]:
drive.EventName.value_counts()

Distance Event                       300366
Timed Event                           37607
Harsh Acceleration                     7168
System Event                           2651
Harsh Braking                          2475
Reached max speed                      2442
Out of max speed                       1102
Network Event                          1068
Harsh Turn (motion based)              1022
Harsh Braking (motion based)            652
Engine started                          545
Engine turned off                       420
Harsh Turn Right (motion based)         381
Harsh Turn Left (motion based)          376
Harsh Acceleration (motion based)       348
Name: EventName, dtype: int64

In [7]:
drp = ['System Event', 'Network Event','Engine turned off', 'Engine started','Distance Event']
rest = list(set(drive.EventName.unique()).difference(set(drp)))
make = drive[drive.EventName.isin(rest)]

In [8]:
make = make.replace({'EventName': {'Harsh Acceleration (motion based)':'Harsh Acceleration',
                       'Harsh Braking (motion based)':'Harsh Braking',
                        'Harsh Acceleration (motion based)':'Harsh Acceleration',
                        'Harsh Turn Left (motion based)':'Harsh Turn',
                        'Harsh Turn Right (motion based)':'Harsh Turn',
                        'Harsh Turn (motion based)':'Harsh Turn',
                        'Reached max speed': 'Beyond max speed',
                        'Out of max speed':'Beyond max speed'}})

make.EventName.value_counts()

Timed Event           37607
Harsh Acceleration     7516
Beyond max speed       3544
Harsh Braking          3127
Harsh Turn             1779
Name: EventName, dtype: int64

In [9]:
make.columns, make.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 53573 entries, 445168 to 111757
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   DriverId    53573 non-null  int64  
 1   EventName   53573 non-null  object 
 2   Latitude    53573 non-null  float64
 3   Longitude   53573 non-null  float64
 4   Speed km/h  53573 non-null  float64
 5   ts          53573 non-null  object 
dtypes: float64(3), int64(1), object(2)
memory usage: 2.9+ MB


(Index(['DriverId', 'EventName', 'Latitude', 'Longitude', 'Speed km/h', 'ts'], dtype='object'),
 None)

In [10]:
# Days of the Week
import datetime
make.ts = pd.to_datetime(make.ts) #.strftime('%d/%m/%Y %H:%M')
make['Days'] =  [t.date().strftime('%A') for t in make.ts]
# dty = ['Saturday', 'Sunday']
# make['Week'] = ['Weekend' if i in dty else 'Weekday' for i in make.Days]

# Visibility
make['visiblity'] = [ 0 if i.time().hour in range(6, 21) else 1 for i in make.ts]

# Aggressive Driving
crz = ['Harsh Acceleration', 'Harsh Braking', 'Harsh Turn', 'Beyond max speed']
make['agg_drv'] = [1 if i in crz else 0 for i in make.EventName]

# Overspeed
# Speed threshold is set at 80 km/h and excess at 100
make['overspeed'] = [1 if make.iloc[i]['Speed km/h'] > 80 else 1 if make.iloc[i]['EventName']=='Harsh Turn' 
                        and make.iloc[i]['Speed km/h'] > 40 else 0 for i in range(len(make))]

# Fatigue Driving
# Fatigue driving is aroung afternoon towards the evening. 1 in 25 drivers suffer this
make['fatigue'] = [rd.choices([1, 0], weights=[1,25], k=1)[0] if i.time().hour in range(12,24) else 0 for i in make.ts] # Self generated

# Possible collision distance
# This is a subraction of stopping distance from car distance:
# Taking reaction time to be 0.75s for normal and 1.5 for fatigued, road coef to asphalt 0.8,
# car distance to 3s
make['col_dist'] = [round((make.iloc[i]['Speed km/h'] *3)/3.6 - ((make.iloc[i]['Speed km/h'] * 1.5)/3.6 + make.iloc[i]['Speed km/h']**2/(250*0.8)), 2)  
                    if make.iloc[i]['fatigue']=='Yes' else round((make.iloc[i]['Speed km/h']*3)/3.6 - ((make.iloc[i]['Speed km/h'] * 0.75)/3.6 + make.iloc[i]['Speed km/h']**2/(250*0.8)), 2) 
                    for i in range(len(make))]

make['col_distd'] = [1 if i < 0 else 0 for i in make.col_dist]



In [11]:
# Based on research, respective weight for each driver behaviour was acquired
make['Fault'] = [round((0.051*make.iloc[i]['visiblity'] + 0.07326*make.iloc[i]['agg_drv']
                    + 0.225 * make.iloc[i]['fatigue'] +  0.0267*make.iloc[i]['overspeed'] + 0.0411*make.iloc[i]['col_distd'])* 10,2)
                    for i in range(len(make))]
make['Decision'] = [1 if i > 2.4 else 0 for i in make.Fault]

In [12]:
make.head(10)

,DriverId,EventName,Latitude,Longitude,Speed km/h,ts,Days,visiblity,agg_drv,overspeed,fatigue,col_dist,col_distd,Fault,Decision
445168,45,Timed Event,33.331807,-118.463322,59.0,2017-11-12 15:48:40.230,Sunday,0,0,0,0,19.47,0,0.00,0
558156,51,Timed Event,33.757704,-118.118819,72.0,2017-11-17 12:47:14.190,Friday,0,0,0,0,19.08,0,0.00,0
112440,8,Timed Event,33.956235,-118.260200,32.0,2017-11-10 13:04:03.130,Friday,0,0,0,0,14.88,0,0.00,0
708040,72,Timed Event,33.695880,-118.487723,61.0,2017-11-09 08:49:01.840,Thursday,0,0,0,0,19.52,0,0.00,0
534930,51,Harsh Acceleration,33.942351,-118.088497,51.0,2017-11-02 09:29:33.080,Thursday,0,1,0,0,18.87,0,0.73,0
588750,55,Timed Event,34.076297,-118.305923,91.0,2017-11-03 08:18:48.900,Friday,0,0,1,0,15.47,0,0.27,0
410291,44,Harsh Acceleration,33.808472,-118.463846,49.0,2017-11-04 23:51:17.110,Saturday,1,1,0,0,18.62,0,1.24,0
109091,8,Timed Event,33.989907,-118.336310,70.0,2017-11-07 14:20:50.670,Tuesday,0,0,0,0,19.25,0,0.00,0
397117,41,Timed Event,34.434713,-118.145270,18.0,2017-11-13 17:37:04.440,Monday,0,0,0,0,9.63,0,0.00,0
356437,35,Timed Event,34.328066,-118.215139,35.0,2017-11-17 19:10:05.140,Friday,0,0,0,0,15.75,0,0.00,0


In [13]:
make = make.drop(['DriverId', 'Longitude', 'Latitude', 'ts'], axis=1)
make.corr() 

,Speed km/h,visiblity,agg_drv,overspeed,fatigue,col_dist,col_distd,Fault,Decision
Speed km/h,1.000000,0.036124,0.200799,0.805240,-0.006894,-0.038884,0.321523,0.341002,0.056961
visiblity,0.036124,1.000000,0.016535,0.019465,0.008551,0.034630,0.000500,0.302627,0.048983
agg_drv,0.200799,0.016535,1.000000,0.073474,0.000142,0.178887,0.044555,0.648018,0.073361
overspeed,0.805240,0.019465,0.073474,1.000000,-0.009746,-0.318930,0.243115,0.289486,0.051675
fatigue,-0.006894,0.008551,0.000142,-0.009746,1.000000,0.005748,-0.002807,0.654381,0.702891
col_dist,-0.038884,0.034630,0.178887,-0.318930,0.005748,1.000000,-0.436874,0.008762,0.009716
col_distd,0.321523,0.000500,0.044555,0.243115,-0.002807,-0.436874,1.000000,0.185292,0.011949
Fault,0.341002,0.302627,0.648018,0.289486,0.654381,0.008762,0.185292,1.000000,0.531982
Decision,0.056961,0.048983,0.073361,0.051675,0.702891,0.009716,0.011949,0.531982,1.000000


In [14]:
# from matplotlib.pylab import rcParams
# rcParams['figure.figsize'] = 20,10

In [15]:
make.fatigue.value_counts()

0    52236
1     1337
Name: fatigue, dtype: int64

In [16]:
make.Fault.value_counts()

0.00    25631
0.73     9679
0.27     7067
1.00     3845
0.51     2503
1.24     1166
0.78      893
2.25      668
0.68      514
1.51      423
1.41      412
2.98      256
2.52      158
3.25       81
2.76       69
1.19       61
1.92       42
3.49       41
3.03       31
2.93       11
3.66       11
3.76       10
3.44        1
Name: Fault, dtype: int64

In [17]:
make.Decision.value_counts()

0    52904
1      669
Name: Decision, dtype: int64

In [18]:
# sns.countplot(x= 'overspeed', data = make, hue ='visiblity')
# sns.jointplot(x = "overspeed", y = "Speed km/h", data= make, kind='reg')

In [19]:
# sns.jointplot(y= "col_dist", x = "Speed km/h", data= make, alpha =0.5)
# sns.barplot(x = 'Days', y='Speed km/h', data =make)
# sns.pairplot(make, hue='visiblity')

In [20]:
# sns.jointplot(x = "col_dist", y = "overspeed", data= make, kind='reg')
# sns.countplot(data = make, x='EventName')

In [21]:
make.EventName.value_counts()

Timed Event           37607
Harsh Acceleration     7516
Beyond max speed       3544
Harsh Braking          3127
Harsh Turn             1779
Name: EventName, dtype: int64

In [22]:
make.sample(20)

,EventName,Speed km/h,Days,visiblity,agg_drv,overspeed,fatigue,col_dist,col_distd,Fault,Decision
807960,Timed Event,51.0,Friday,1,0,0,0,18.87,0,0.51,0
272241,Timed Event,87.0,Saturday,0,0,1,0,16.53,0,0.27,0
445216,Timed Event,87.0,Sunday,0,0,1,0,16.53,0,0.27,0
417038,Harsh Acceleration,51.0,Monday,0,1,0,0,18.87,0,0.73,0
742902,Timed Event,29.0,Thursday,1,0,0,0,13.92,0,0.51,0
77640,Harsh Acceleration,23.0,Thursday,0,1,0,0,11.73,0,0.73,0
495448,Timed Event,37.0,Wednesday,0,0,0,0,16.28,0,0.00,0
118292,Harsh Acceleration,93.0,Thursday,0,1,1,0,14.88,0,1.00,0
360354,Timed Event,20.0,Friday,0,0,0,0,10.50,0,0.00,0
38861,Timed Event,86.0,Wednesday,0,0,1,0,16.77,0,0.27,0


In [23]:
make.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 53573 entries, 445168 to 111757
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EventName   53573 non-null  object 
 1   Speed km/h  53573 non-null  float64
 2   Days        53573 non-null  object 
 3   visiblity   53573 non-null  int64  
 4   agg_drv     53573 non-null  int64  
 5   overspeed   53573 non-null  int64  
 6   fatigue     53573 non-null  int64  
 7   col_dist    53573 non-null  float64
 8   col_distd   53573 non-null  int64  
 9   Fault       53573 non-null  float64
 10  Decision    53573 non-null  int64  
dtypes: float64(3), int64(6), object(2)
memory usage: 4.9+ MB


In [24]:
# Importing packages
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as mse
# from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost.sklearn import XGBClassifier 

In [25]:
make.skew()

Speed km/h    0.501060
visiblity     2.707892
agg_drv       0.883196
overspeed     1.135682
fatigue       6.090750
col_dist     -1.067648
col_distd     6.924422
Fault         2.069455
Decision      8.780443
dtype: float64

In [26]:
maked = make.drop(['overspeed', 'col_distd', 'Fault' ], axis=1)

In [29]:
# stroke = stroke.drop('id' , axis =1)
train,test = train_test_split(maked, test_size = 0.2, random_state = 101)
X = train.drop(['Decision'], axis=1)
y = train.Decision

In [30]:
# sk = [i for i in new.columns if abs(new[i].skew()) > 1]
# for i in sk:
#     new[i] = new[i].apply(lambda i: np.log(i) if i > 0 else 0)

In [31]:
trainX,testX, trainy,testy= train_test_split(X, y, test_size = 0.2, random_state = 101)

In [34]:
import category_encoders as ce
encode = ce.TargetEncoder(cols= ['Days','EventName'])
trainX = encode.fit_transform(trainX, trainy)
testX = encode.transform(testX)

sc = StandardScaler()
trainX = sc.fit_transform(trainX)
testX = sc.transform(testX)

In [35]:
# Finding the BaseLine perfomance of the various models

# prepare models
models = []

# Adding algorthms
models.append(('sgd', SGDClassifier()))
models.append(('LR', LogisticRegression()))
# models.append(('knr', KNeighborsClassifier()))
models.append(('rfr', RandomForestClassifier()))
models.append(('xg', XGBClassifier(objective ='reg:squarederror')))
# evaluate -cross validation- each model in turn
results = []
names = []
scoring =['accuracy', 'f1']
for name, model in models:
	kfold = KFold(n_splits=10)
	cv_results = cross_validate(model, trainX, trainy, cv=kfold, scoring=scoring, return_train_score=True)
	results.append(cv_results)
	names.append(name)
	msg = "%s: %f (%f)" % (name, cv_results['train_accuracy'].mean(), cv_results['test_f1'].mean())
	print(msg)


sgd: 0.999469 (0.975947)
LR: 0.999851 (0.992498)
rfr: 1.000000 (0.998630)
xg: 1.000000 (1.000000)
